In [1]:
import os
import xarray as xr
import yaml
import importlib

import fates_calibration_library.param_ens_gen.parameter as parameter
import fates_calibration_library.param_ens_gen.param_ensemble as param_ensemble
import fates_calibration_library.param_ens_gen.sampler as sampler

In [16]:
importlib.reload(parameter)

<module 'fates_calibration_library.param_ens_gen.parameter' from '/glade/work/afoster/FATES_calibration/fates_calibration_library/fates_calibration_library/param_ens_gen/parameter.py'>

In [2]:
param_dir = '/glade/work/afoster/FATES_calibration/test_param_gen'

param_sheet_file = 'param_list_test.xls'
default_ds_file = 'fates_params_api40_nwt_update_agb.nc'
posterior_sources_file = 'posterior_sources.yaml'

param_data_file = os.path.join(param_dir, param_sheet_file)
default_ds = xr.open_dataset(os.path.join(param_dir, default_ds_file))
posterior_sources = os.path.join(param_dir, posterior_sources_file)

In [ ]:
# ens = ParamEnsemble.from_dict({
#     'ensemble_type': 'LatinHypercube',
#     'param_data_file': os.path.join(param_dir, param_sheet_file),
#     'ensemble_dir': 'test_LH',
#     'file_prefix': 'my_LH',
#     'default_param_file': os.path.join(param_dir, default_ds_file),
#     'fixed_indices': {'fates_pft': [5, 7, 8, 9, 10, 11, 12, 13]},
#     'ensemble_members': 3,
#     'posterior_sources': os.path.join(param_dir, posterior_sources_file),
# })
# ens.create_ensemble()

In [3]:
main, pft_sheets = param_ensemble._read_param_list(param_data_file)

with open(posterior_sources, "r", encoding="utf-8") as f:
    posterior_config = yaml.safe_load(f)

In [17]:
params = [
        parameter.Parameter.from_row(
            row,
            pft_sheet=pft_sheets.get(row["parameter_name"]),
            default_ds=default_ds,
            posterior_config=posterior_config.get(row["parameter_name"]),
        )
        for _, row in main.iterrows()
    ]

In [25]:
for param in params:
    default_value = param.get_default(default_ds)
    mask = None
    array_index = param.active_index.index if param.active_index is not None else None
    value = param.sampler.sample(
        0.5, mask, default_value, array_index, param.n_indices
    )

In [11]:
ds = default_ds.copy(deep=False)

In [12]:
param.set_value(
        ds,
        default_ds,
        value,
    )

In [13]:
ds[param.spec.name].values

array([[0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25,
        0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25,
        0.25, 0.25, 0.25],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
        0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
        0.  , 0.  , 0.  ]])